# Guía para usar el dashboard en Tableau Public
## Grupo 2 — Ángel Espín · Carlos Ramírez

---

### ¿Por qué este cuaderno?

El cuaderno principal (`Grupo2_Retail_Sales_Documentacion.ipynb`) genera un archivo
**`retail_sales.hyper`** para conectar Tableau. El formato `.hyper` es el extracto nativo de
**Tableau Desktop**, pero **Tableau Public** (la versión gratuita) **no puede leer archivos `.hyper`**
directamente.

Sin embargo, Tableau Public **sí puede conectarse a archivos CSV, Excel o Google Sheets**.
El dataset limpio ya existe como `retail_sales_clean.csv` con **los mismos 17 atributos**,
incluyendo todos los derivados (Mes, Trimestre, GrupoEdad, NivelPrecio…).

**Este cuaderno explica cómo lograr exactamente el mismo dashboard directamente desde
Tableau Public usando `retail_sales_clean.csv` como origen de datos.**

---

### Diferencia clave entre `.hyper` y `.csv` en Tableau

| Aspecto | `.hyper` (Desktop) | `.csv` (Public) |
|---|---|---|
| Lectura en Tableau Public | ❌ No compatible | ✅ Compatible |
| Velocidad de carga | Más rápida (formato columnar nativo) | Más lenta (texto plano) |
| Actualización de datos | Requiere regenerar el `.hyper` | Tableau Public la pide al publicar |
| Tamaño | Menor (comprimido) | Mayor (texto sin comprimir) |
| Contiene datos | Sí (es un extracto) | Sí (es el origen) |
| Se necesita Python | Sí (para generarlo) | No (es el CSV ya limpio) |

**Conclusión:** para Tableau Public, usa directamente `retail_sales_clean.csv`.
Los pasos de construcción del dashboard son **idénticos** a los del Anexo C del cuaderno principal.


---

## Paso 0: Verifica que `retail_sales_clean.csv` está en tu proyecto

El archivo ya fue generado por el Anexo A del cuaderno principal. Debe estar en:
```
proyecto/retail_sales_clean.csv
```
Si no está, ejecuta las celdas A.1 a A.3 del cuaderno principal para generarlo.

**Columnas disponibles (17):**
- Originales: `Transaction ID`, `Date`, `Customer ID`, `Gender`, `Age`,
  `Product Category`, `Quantity`, `Price per Unit`, `Total Amount`
- Derivadas: `Anio`, `Mes`, `NombreMes`, `Trimestre`, `DiaSemana`,
  `TipoDia`, `GrupoEdad`, `NivelPrecio`

Todas las derivaciones que en el cuaderno original se hacían con Python
(`Anio`, `Mes`, `NombreMes`, etc.) **ya están calculadas en el CSV**.
No necesitas crear campos calculados en Tableau para ellas.

---

## Paso 1: Conectar Tableau Public al CSV

1. Abre **Tableau Public** (descárgalo gratis de https://public.tableau.com si no lo tienes).
2. En la pantalla de inicio, en el panel izquierdo **Connect → To a File → Text File**.
3. Navega hasta `retail_sales_clean.csv` y selecciónalo.
4. Tableau mostrará una vista previa. Asegúrate de que:
   - **Separador:** Coma (`,`).
   - **Codificación:** UTF-8.
   - **First row as header:** ✅ marcado.
5. Haz clic en **Sheet 1** (hoja de trabajo) para empezar.

### Ajustar tipos de datos automáticamente

Tableau debería detectar automáticamente:
- **Dimensiones** (categorías/fechas): `Date`, `NombreMes`, `Trimestre`, `DiaSemana`,
  `TipoDia`, `Product Category`, `Gender`, `GrupoEdad`, `NivelPrecio`, `Customer ID`,
  `Transaction ID`, `Anio`, `Mes`.
- **Medidas** (números a agregar): `Total Amount`, `Quantity`, `Price per Unit`, `Age`.

> ⚠️ **Verifica:** si `Mes`, `Anio` o `Transaction ID` aparecen como **medida** (en la sección
> de arriba con el símbolo `#`), haz clic derecho → **Convert to Dimension**.
> No queremos sumar identificadores ni ordinales.

> ⚠️ **Verifica:** `Date` debe aparecer como **fecha** (símbolo de calendario). Si no,
> haz clic derecho → **Change Data Type → Date**.

---

## Paso 2: Crear los 5 worksheets (hojas)

A continuación se resumen los pasos. Para una explicación detallada de **cada vista**,
consulta el **Anexo C** del cuaderno principal (`Grupo2_Retail_Sales_Documentacion.ipynb`).
Las instrucciones son **exactamente las mismas**; solo cambia el origen de datos.

---

### WS-1: KPIs (tarjetas numéricas)

Crea **4 hojas** (o usa *text marks*). Para cada KPI:
- **Marks → tipo:** Text.
- Arrastra la medida correspondiente a **Text** en el panel Marks.

| KPI | Campo a arrastrar | Agregación |
|---|---|---|
| Ingresos totales | `Total Amount` | `SUM` |
| Transacciones | `Transaction ID` | `CNT` (o `COUNT`) |
| Ticket promedio | Campo calculado (ver abajo) | — |
| Unidades vendidas | `Quantity` | `SUM` |

**Ticket promedio** — crea un **campo calculado**:
1. *Analysis → Create Calculated Field*.
2. Nombre: `Ticket promedio`.
3. Fórmula:
   ```
   SUM([Total Amount]) / COUNT([Transaction ID])
   ```
4. Arrástralo a **Text** en una nueva hoja.

> 💡 **Formato:** Da clic derecho al número → **Format** → elige un tamaño grande (18-24 pt)
> y un título corto (ej. "Ingresos totales").

---

### WS-2: Tendencia mensual (gráfico de líneas)

| Elemento | Colocar en |
|---|---|
| `NombreMes` | Columns |
| `SUM(Total Amount)` | Rows |
| *(opcional)* `Product Category` | Color (Marks) |

- **Marks → tipo:** **Line**.
- Activa **marcadores** (puntos) en el menú de la tarjeta Marks.
- `NombreMes` se ordena automáticamente porque tiene prefijo numérico (`01-Enero`, `02-Febrero`…).
- Añade **título** a la hoja: "Ingresos por mes (2023)".

**Variante opcional:** arrastra `Product Category` a Color para ver 3 líneas separadas
(Beauty, Clothing, Electronics) con la misma paleta que las otras vistas.

---

### WS-3: Ingresos por categoría (barras ordenadas)

| Elemento | Colocar en |
|---|---|
| `Product Category` | Rows |
| `SUM(Total Amount)` | Columns |
| `Product Category` | Color (Marks) |

- **Marks → tipo:** **Bar**.
- **Ordena descendente** por `SUM(Total Amount)`: haz clic en el ícono de *sort* en la barra
  de herramientas (o en el propio eje).
- Activa **Label** para mostrar el valor de cada barra.
- Usa paleta **Tableau 10** (colores 100% distinguibles).

---

### WS-4: Perfil demográfico (barras agrupadas: edad × género)

| Elemento | Colocar en |
|---|---|
| `GrupoEdad` | Columns |
| `Gender` | Columns (junto a GrupoEdad) |
| `SUM(Total Amount)` | Rows |
| `Gender` | Color (Marks) |

- **Marks → tipo:** **Bar**.
- Al poner `Gender` junto a `GrupoEdad` en Columns, Tableau automáticamente crea **barras agrupadas**
  (*side-by-side*).
- Elige colores distintos para Female y Male (ej. rosado/azul, o dos tonos de Tableau 10).
- Título: "Ingresos por grupo de edad y género".

---

### WS-5: Mapa de calor Categoría × Mes (highlight table)

| Elemento | Colocar en |
|---|---|
| `NombreMes` | Columns |
| `Product Category` | Rows |
| `SUM(Total Amount)` | Color (Marks) |

- **Marks → tipo:** **Square** (cuadro).
- Haz clic en **Color** dentro de Marks → elige una paleta **secuencial** (ej. *Blue* o *Orange*).
- *(Opcional)* arrastra `SUM(Total Amount)` también a **Label** para ver los valores dentro de
  cada celda.
- Si las columnas están en orden alfabético en vez de cronológico, arrastra `NombreMes`
  manualmente para reordenarlas de enero a diciembre.

---

## Paso 3: Armar el Dashboard

1. **Dashboard → New Dashboard**.
2. Elige un tamaño (p. ej. 1200×800 o *Automatic*).
3. Arrastra las hojas en este orden sugerido:

   ```
   ┌────────────────────────────────────────────────┐
   │      KPI 1    KPI 2    KPI 3    KPI 4          │  ← fila de KPIs
   ├──────────────────────┬─────────────────────────┤
   │                      │                         │
   │  Tendencia mensual   │  Ingresos por categoría │
   │  (líneas)            │  (barras)               │
   │                      │                         │
   ├──────────────────────┼─────────────────────────┤
   │                      │                         │
   │  Perfil demográfico  │  Mapa de calor          │
   │  (barras agrupadas)  │  Categoría × Mes        │
   │                      │                         │
   └──────────────────────┴─────────────────────────┘
   ```

### Filtros interactivos

En cualquiera de las hojas, haz clic derecho sobre una dimensión y selecciona **Show Filter**.
Recomendamos filtrar por:
- `Trimestre` (T1, T2, T3, T4)
- `Gender` (Female / Male)
- `Product Category` (Beauty / Clothing / Electronics)

Para que un filtro afecte a **todo el dashboard**:
1. Muestra el filtro.
2. Haz clic en el menú desplegable del filtro → **Apply to Worksheets →
   All Using This Data Source**.

### Título e integrantes (requisito de las Fases 1 y 2)

1. Arrastra un objeto **Text** a la parte superior del dashboard.
2. Escribe:
   > **Análisis Visual de Ventas Minoristas 2023**
3. Arrastra **otro objeto Text** debajo con:
   > **Grupo 2 — Ángel Espín & Carlos Ramírez**
4. Opcional: añade un texto al pie:
   > *Fuente: Retail Sales Dataset (Kaggle). Datos sintéticos, 2023.*

---

## Paso 4: Publicar en Tableau Public

1. **Server → Tableau Public → Save to Tableau Public As…**
2. Inicia sesión o crea una cuenta gratuita.
3. Ponle nombre al dashboard (ej. "Retail Sales Dashboard — Grupo 2").
4. Tableau Public te dará un enlace público para compartir.

> ⚠️ **Importante:** Tableau Public te preguntará si quieres extraer los datos. Como el CSV ya
> es un extracto de texto plano, confirma la extracción. Tableau lo empaquetará en su propio
> formato interno automáticamente.

### Alternativa: guardar como `.twbx` para Desktop

Si en el futuro alguien tiene Tableau Desktop:
- **File → Save As… → Tableau Packaged Workbook (`.twbx`)**.
- El `.twbx` incluirá el CSV dentro, listo para abrir en Desktop.

---

## Notas importantes sobre diferencias con el `.hyper`

1. **Rendimiento:** Tableau Public con CSV puede ser un poco más lento que con `.hyper` en
   Desktop. Con solo 1000 filas la diferencia es imperceptible.

2. **Actualización de datos:** En Tableau Public, si quisieras actualizar los datos, tendrías que
   subir un nuevo CSV. En Desktop con `.hyper`, regenarías el `.hyper` con Python y lo
   reemplazarías.

3. **No se necesita Python para Public:** Una vez que el CSV está limpio y tiene los atributos
   derivados, cualquier persona puede abrir Tableau Public, conectar el CSV y construir el
   dashboard sin escribir una línea de código.

4. **Mismo dashboard, misma documentación:** Las Fases 1-4, la justificación de diseño
   (marcas y canales), las tareas abstractas y el escenario de uso **son idénticos**,
   independientemente de si usas `.hyper` o `.csv` como origen.

---

## Referencias

- Cuaderno principal: `Grupo2_Retail_Sales_Documentacion.ipynb`
- Dataset limpio: `retail_sales_clean.csv`
- Tableau Public: https://public.tableau.com
- Datos originales: https://www.kaggle.com/datasets/mohammadtalib786/retail-sales-dataset

---

*Generado el 2026-06-07 para el proyecto "Análisis Visual de Ventas Minoristas" — Grupo 2*